In [1]:
#importing dependencis
import pyspark
from pyspark.sql import SparkSession
#creating spark session
spark=SparkSession.builder\
                    .appName("FIFA_project")\
                    .getOrCreate()
#uploading data and creating df
df=spark.read.option("header",True).option("inferSchema",True).csv("FIFA_project/historical/results.csv")
tie_breaker=spark.read.option("header",True).option("inferSchema",True).csv("FIFA_project/historical/shootouts.csv")
goal_scorers=spark.read.option("header",True).option("inferSchema",True).csv("FIFA_project/historical/goalscorers.csv")
#display df
df.show(5)
tie_breaker.limit(5).show()
goal_scorers.limit(5).show()

+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|      date|home_team|away_team|home_score|away_score|tournament|   city| country|neutral|
+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|1872-11-30| Scotland|  England|         0|         0|  Friendly|Glasgow|Scotland|  false|
|1873-03-08|  England| Scotland|         4|         2|  Friendly| London| England|  false|
|1874-03-07| Scotland|  England|         2|         1|  Friendly|Glasgow|Scotland|  false|
|1875-03-06|  England| Scotland|         2|         2|  Friendly| London| England|  false|
|1876-03-04| Scotland|  England|         3|         0|  Friendly|Glasgow|Scotland|  false|
+----------+---------+---------+----------+----------+----------+-------+--------+-------+
only showing top 5 rows

+----------+-----------+----------------+-----------+-------------+
|      date|  home_team|       away_team|     winner|first_shooter|
+----------+--------

In [2]:
#identifying tournaments
from pyspark.sql.functions import desc
df.groupBy("tournament").count().orderBy(desc("count")).show(truncate=False)

+------------------------------------+-----+
|tournament                          |count|
+------------------------------------+-----+
|Friendly                            |18252|
|FIFA World Cup qualification        |8771 |
|UEFA Euro qualification             |2824 |
|African Cup of Nations qualification|2327 |
|FIFA World Cup                      |1036 |
|Copa América                        |869  |
|African Cup of Nations              |845  |
|AFC Asian Cup qualification         |829  |
|UEFA Nations League                 |658  |
|CECAFA Cup                          |620  |
|CFU Caribbean Cup qualification     |606  |
|Merdeka Tournament                  |599  |
|British Home Championship           |523  |
|CONCACAF Nations League             |422  |
|AFC Asian Cup                       |421  |
|Gold Cup                            |420  |
|Gulf Cup                            |410  |
|Island Games                        |394  |
|UEFA Euro                           |388  |
|Asian Gam

In [3]:
#creating copy for handling 2022 to 2026 data
df_recent=df.alias("df_recent")

##Handling the World Cup data till 2022 FIFA WC

In [4]:
#creating df only for WC
from pyspark.sql.functions import col,year
df_wc=df.filter((col("tournament")=="FIFA World Cup") & (year(col("date"))>=1990) & (year(col("date"))<2023))
#creating a year column
df_wc=df_wc.withColumn("year",year(col("date")))
#checking schema
df_wc.printSchema()

root
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_score: string (nullable = true)
 |-- away_score: string (nullable = true)
 |-- tournament: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- neutral: boolean (nullable = true)
 |-- year: integer (nullable = true)



In [5]:
#changing the dtype
df_wc=df_wc.withColumn("home_score",col("home_score").cast("int"))
df_wc=df_wc.withColumn("away_score",col("away_score").cast("int"))
df_wc.printSchema()

root
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_score: integer (nullable = true)
 |-- away_score: integer (nullable = true)
 |-- tournament: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- neutral: boolean (nullable = true)
 |-- year: integer (nullable = true)



In [6]:
#Checking the national teams
df_wc.groupBy("home_team").count().show(10)

+-----------+-----+
|  home_team|count|
+-----------+-----+
|     Russia|    9|
|   Paraguay|    6|
|    Senegal|    4|
|     Sweden|   13|
|     Turkey|    1|
|    Germany|   39|
|Ivory Coast|    3|
|     France|   24|
|     Greece|    4|
|       Togo|    2|
+-----------+-----+
only showing top 10 rows



In [7]:
#changing the former name to current team name
from pyspark.sql.functions import when
df_wc=df_wc.withColumn("home_team",when(col("home_team")=="Czechoslovakia","Czech Republic")
                      .when(col("home_team")=="Yugoslavia","Serbia")
                      .otherwise(col("home_team")))
df_wc=df_wc.withColumn("away_team",when(col("away_team")=="Czechoslovakia","Czech Republic")
                      .when(col("away_team")=="Yugoslavia","Serbia")
                      .otherwise(col("away_team")))
tie_breaker=tie_breaker.withColumn("home_team",when(col("home_team")=="Czechoslovakia","Czech Republic")
                      .when(col("home_team")=="Yugoslavia","Serbia")
                      .otherwise(col("home_team")))
tie_breaker=tie_breaker.withColumn("away_team",when(col("away_team")=="Czechoslovakia","Czech Republic")
                      .when(col("away_team")=="Yugoslavia","Serbia")
                      .otherwise(col("away_team")))
goal_scorers=goal_scorers.withColumn("home_team",when(col("home_team")=="Czechoslovakia","Czech Republic")
                      .when(col("home_team")=="Yugoslavia","Serbia")
                      .otherwise(col("home_team")))
goal_scorers=goal_scorers.withColumn("away_team",when(col("away_team")=="Czechoslovakia","Czech Republic")
                      .when(col("away_team")=="Yugoslavia","Serbia")
                      .otherwise(col("away_team")))
goal_scorers=goal_scorers.withColumn("team",when(col("team")=="Czechoslovakia","Czech Republic")
                      .when(col("team")=="Yugoslavia","Serbia")
                      .otherwise(col("team")))

In [8]:
#creating the winner col
df_wc=df_wc.withColumn("90_min_winner",(when(col("home_score")>col("away_score"),col("home_team"))\
                                .when(col("away_score")>col("home_score"),col("away_team"))\
                                 .otherwise("draw")))

In [9]:
#integrating tie_breaker_winner in df
df_wc=df_wc.join(tie_breaker,on=["date","home_team","away_team"],how="left")

In [10]:
#creating a column containing final match result
df_wc=df_wc.withColumn("match_winner",when((col("90_min_winner")=="draw")&(col("winner").isNotNull()),col("winner"))\
                                        .otherwise(col("90_min_winner")))

In [11]:
#trimming extra columns
df_wc=df_wc.drop("90_min_winner","winner","first_shooter")

In [12]:
#creating yearwise tournament winner table 
#creating a window function
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
#defining the window blueprint
window_spec=Window.partitionBy("year").orderBy(col("date").desc())
#applying it to dataframe to get desired output
tournament_winners=df_wc.withColumn("rank",row_number().over(window_spec))\
                        .filter(col("rank")==1)\
                        .select(col("year"),col("match_winner").alias("tournament_winner"))
tournament_winners.show()
#updating main df with tournament winners
df_wc=df_wc.join(tournament_winners,on="year",how="left")
df_wc.limit(2).toPandas()

+----+-----------------+
|year|tournament_winner|
+----+-----------------+
|1990|          Germany|
|1994|           Brazil|
|1998|           France|
|2002|           Brazil|
|2006|            Italy|
|2010|            Spain|
|2014|          Germany|
|2018|           France|
|2022|        Argentina|
+----+-----------------+



,year,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_winner,tournament_winner
0,1990,1990-06-08,Argentina,Cameroon,0,1,FIFA World Cup,Milan,Italy,True,Cameroon,Germany
1,1990,1990-06-09,Italy,Austria,1,0,FIFA World Cup,Rome,Italy,False,Italy,Germany


In [13]:
##Top scorer identification
#distiinct worldcup matches
wc_match_unique=df_wc.select("date","home_team","away_team").distinct()
#goal scored only for worldcup matches
wc_goals=goal_scorers.join(wc_match_unique,on=["date","home_team","away_team"],how="inner")
#adding a year column
wc_goals=wc_goals.withColumn("year",year(col("date")))
#year wise tournament total goals per player
from pyspark.sql.functions import count as spark_count
seasonal_player_goals=wc_goals.groupBy("year","team","scorer").agg(spark_count("scorer").alias("total_goals"))
#creating the window function blueprint
player_window=Window.partitionBy("year","team").orderBy(col("total_goals").desc())
team_window = Window.partitionBy("year", "team")
#creating top players ranking with window function
from pyspark.sql.functions import sum as spark_sum, round as spark_round
players_ranking=seasonal_player_goals.withColumn("rank",row_number().over(player_window))\
                                        .withColumn("total_team_goals",spark_sum("total_goals").over(team_window))
players_ranking=players_ranking.withColumn("contribution",spark_round(100*col("total_goals")/col("total_team_goals"),2))
top_players=players_ranking.filter(col("rank")==1).select(
                                                 col("year"),
                                                 col("team"),
                                                 col("scorer").alias("highest_goal_scorer"),
                                                 col("total_goals").alias("highest_goal_count"),
                                                col("total_team_goals"),
                                                col("contribution"))                                  
top_players.show(10)

+----+--------------+-------------------+------------------+----------------+------------+
|year|          team|highest_goal_scorer|highest_goal_count|total_team_goals|contribution|
+----+--------------+-------------------+------------------+----------------+------------+
|1990|     Argentina|   Claudio Caniggia|                 2|               5|        40.0|
|1990|       Austria|      Andreas Ogris|                 1|               2|        50.0|
|1990|       Belgium|      Lei Clijsters|                 1|               6|       16.67|
|1990|        Brazil|             Careca|                 2|               4|        50.0|
|1990|      Cameroon|        Roger Milla|                 4|               7|       57.14|
|1990|      Colombia|     Bernardo Redín|                 2|               4|        50.0|
|1990|    Costa Rica|       Juan Cayasso|                 1|               4|        25.0|
|1990|Czech Republic|     Tomáš Skuhravý|                 5|              10|        50.0|

In [14]:
#preparing the masterdf
from pyspark.sql.functions import when,sum as spark_sum,avg as spark_avg,first
#preparing home side
home_stats=df_wc.select(
                    col("date"),
                    col("year"),
                    col("home_team").alias("team"),
                    when(col("match_winner")==col("home_team"),1).otherwise(0).alias("win"),
                    when(col("match_winner")==col("away_team"),1).otherwise(0).alias("loss"),
                    when(col("match_winner")=="draw",1).otherwise(0).alias("draw"),
                    col("home_score").alias("goals_scored"),
                    col("away_score").alias("goals_conceded"),
                    when(col("away_score") == 0, 1).otherwise(0).alias("clean_sheet"),
                    col("tournament_winner")
                    )
#home_stats=home_stats.withColumn("avg_goal_difference",avg("goal_margin").over(partionBy("year","team").orderBy("year")))
away_stats=df_wc.select(
                    col("date"),
                    col("year"),
                    col("away_team").alias("team"),
                    when(col("match_winner")==col("away_team"),1).otherwise(0).alias("win"),
                    when(col("match_winner")==col("home_team"),1).otherwise(0).alias("loss"),
                    when(col("match_winner")=="draw",1).otherwise(0).alias("draw"),
                    col("away_score").alias("goals_scored"),
                    col("home_score").alias("goals_conceded"),
                    when(col("home_score") == 0, 1).otherwise(0).alias("clean_sheet"),
                    col("tournament_winner")
                    ) 
#overall team matches on a unified direction/single team perspective
unified_team_stats=home_stats.union(away_stats)
#seasonwise team performance
seasonal_team_performance= unified_team_stats.groupBy("year","team").agg(
    spark_count("team").alias("played"),
    spark_sum("win").alias("won"),
    spark_sum("loss").alias("loss"),
    spark_sum("draw").alias("drawn"),
    spark_sum("goals_scored").alias("goal_count"),
    spark_round(spark_avg(col("goals_scored") - col("goals_conceded")),2).alias("goal_difference_avg"),
    spark_sum("clean_sheet").alias("clean_sheet"),
    spark_round(100*spark_sum(col("clean_sheet"))/spark_count("*"),2).alias("clean_sheet_percentage"),
    spark_round(spark_avg("goals_conceded"),2).alias("av_goal_consumed"),
    first("tournament_winner").alias("tournament_winner")
)
seasonal_team_performance.limit(2).toPandas()

,year,team,played,won,loss,drawn,goal_count,goal_difference_avg,clean_sheet,clean_sheet_percentage,av_goal_consumed,tournament_winner
0,1990,Argentina,7,4,2,1,5,0.14,3,42.86,0.57,Germany
1,1990,Austria,3,1,2,0,2,-0.33,0,0.00,1.00,Germany


In [15]:
#keepimng only semifinalists
semifinalist_window=Window.partitionBy("year").orderBy(col("date").desc())
df_wc_semifinalists=df_wc.withColumn("rank",row_number().over(semifinalist_window))
df_wc_semis=df_wc_semifinalists.filter(col("rank")<=2)
df_wc_semis.limit(10).toPandas()

,year,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_winner,tournament_winner,rank
0,1990,1990-07-08,Germany,Argentina,1,0,FIFA World Cup,Rome,Italy,True,Germany,Germany,1
1,1990,1990-07-07,Italy,England,2,1,FIFA World Cup,Bari,Italy,False,Italy,Germany,2
2,1994,1994-07-17,Brazil,Italy,0,0,FIFA World Cup,Pasadena,United States,True,Brazil,Brazil,1
3,1994,1994-07-16,Sweden,Bulgaria,4,0,FIFA World Cup,Pasadena,United States,True,Sweden,Brazil,2
4,1998,1998-07-12,France,Brazil,3,0,FIFA World Cup,Saint-Denis,France,False,France,France,1
5,1998,1998-07-11,Netherlands,Croatia,1,2,FIFA World Cup,Paris,France,True,Croatia,France,2
6,2002,2002-06-30,Germany,Brazil,0,2,FIFA World Cup,Yokohama,Japan,True,Brazil,Brazil,1
7,2002,2002-06-29,South Korea,Turkey,2,3,FIFA World Cup,Daegu,South Korea,False,Turkey,Brazil,2
8,2006,2006-07-09,Italy,France,1,1,FIFA World Cup,Berlin,Germany,True,Italy,Italy,1
9,2006,2006-07-08,Germany,Portugal,3,1,FIFA World Cup,Stuttgart,Germany,False,Germany,Italy,2


In [16]:
#keeping only semi finalist teams
master_df = df_wc_semis.select("year", col("home_team").alias("team")).union(
    df_wc_semis.select("year", col("away_team").alias("team"))
).distinct().join(seasonal_team_performance, on=["year", "team"], how="inner")
master_df.limit(10).toPandas()

,year,team,played,won,loss,drawn,goal_count,goal_difference_avg,clean_sheet,clean_sheet_percentage,av_goal_consumed,tournament_winner
0,1990,Argentina,7,4,2,1,5,0.14,3,42.86,0.57,Germany
1,1990,England,7,3,2,2,8,0.29,3,42.86,0.86,Germany
2,1990,Germany,7,6,0,1,15,1.43,2,28.57,0.71,Germany
3,1990,Italy,7,6,1,0,10,1.14,5,71.43,0.29,Germany
4,1994,Brazil,7,6,0,1,11,1.14,5,71.43,0.43,Brazil
5,1994,Bulgaria,7,4,3,0,10,-0.14,2,28.57,1.57,Brazil
6,1994,Italy,7,4,2,1,8,0.43,2,28.57,0.71,Brazil
7,1994,Sweden,7,4,1,2,15,1.00,1,14.29,1.14,Brazil
8,1998,Brazil,7,5,2,0,14,0.57,1,14.29,1.43,France
9,1998,Croatia,7,5,2,0,11,0.86,3,42.86,0.71,France


In [17]:
#joining two dfs(seasonal_team_performance & seasonal_top_scorers)
master_historical_df=master_df.join(top_players, on=["year", "team"], how="left")
master_historical_df.limit(36).toPandas()

,year,team,played,won,loss,drawn,goal_count,goal_difference_avg,clean_sheet,clean_sheet_percentage,av_goal_consumed,tournament_winner,highest_goal_scorer,highest_goal_count,total_team_goals,contribution
0,1990,Argentina,7,4,2,1,5,0.14,3,42.86,0.57,Germany,Claudio Caniggia,2,5,40.00
1,1990,England,7,3,2,2,8,0.29,3,42.86,0.86,Germany,Gary Lineker,4,8,50.00
2,1990,Germany,7,6,0,1,15,1.43,2,28.57,0.71,Germany,Lothar Matthäus,4,15,26.67
3,1990,Italy,7,6,1,0,10,1.14,5,71.43,0.29,Germany,Salvatore Schillaci,6,10,60.00
4,1994,Brazil,7,6,0,1,11,1.14,5,71.43,0.43,Brazil,Romário,5,11,45.45
5,1994,Bulgaria,7,4,3,0,10,-0.14,2,28.57,1.57,Brazil,Hristo Stoichkov,6,10,60.00
6,1994,Italy,7,4,2,1,8,0.43,2,28.57,0.71,Brazil,Roberto Baggio,5,8,62.50
7,1994,Sweden,7,4,1,2,15,1.00,1,14.29,1.14,Brazil,Kennet Andersson,5,15,33.33
8,1998,Brazil,7,5,2,0,14,0.57,1,14.29,1.43,France,Ronaldo,4,14,28.57
9,1998,Croatia,7,5,2,0,11,0.86,3,42.86,0.71,France,Davor Šuker,6,11,54.55


##Handling The data after the 2022 WC

In [18]:
#df_recent handling
df_recent=df_recent.filter((col("date")>"2022-12-18") & (col("date")<"2026-06-11"))
#null values check
null_values=[df_recent.filter(col(col_name).isNull()).count() for col_name in df_recent.columns]
null_values

[0, 0, 0, 0, 0, 0, 0, 0, 0]

In [19]:
#identifying draw matches and handling them
df_recent=df_recent.withColumn("90_minutes_winner",when(col("home_score")>col("away_score"),col("home_team"))
                              .when(col("away_score")>col("home_score"),col("away_team"))
                              .otherwise("draw"))
df_recent=df_recent.join(tie_breaker,how="left",on=["date","home_team","away_team"])
df_recent=df_recent.withColumn("match_winner",when((col("90_minutes_winner")=="draw")&(col("winner").isNotNull()),col("winner"))\
                                        .otherwise(col("90_minutes_winner")))
df_recent=df_recent.drop("90_minutes_winner","winner","first_shooter")
df_recent.limit(1).toPandas()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_winner
0,2022-12-20,Cambodia,Philippines,3,2,AFF Championship,Phnom Penh,Cambodia,False,Cambodia


In [20]:
#goal counts for recent season
#unique matches after 2022 season
recent_unique_matches=df_recent.select("date","home_team","away_team").distinct()
#merging with goal scorers
df_recent_goal_scorers=goal_scorers.join(recent_unique_matches,how="inner",on=["date","home_team","away_team"])
#adding the year column
df_recent_goal_scorers=df_recent_goal_scorers.withColumn("year",year(col("date")))


In [21]:
##preparing the recent dataset to compare the historical dataset
#home games data
home_stats_recent=df_recent.select(
    col("home_team").alias("team"),
    when(col("match_winner")==col("home_team"),1).otherwise(0).alias("Win"),
    when(col("match_winner")==col("away_team"),1).otherwise(0).alias("loss"),
    when(col("match_winner")=="draw",1).otherwise(0).alias("draw"),
    col("home_score").alias("goal_scored"),
    col("away_score").alias("goals_conceded"),
    when(col("away_score") == 0, 1).otherwise(0).alias("clean_sheet"))
#away games data
away_stats_recent=df_recent.select(
    col("away_team").alias("team"),
    when(col("match_winner")==col("away_team"),1).otherwise(0).alias("Win"),
    when(col("match_winner")==col("home_team"),1).otherwise(0).alias("loss"),
    when(col("match_winner")=="draw",1).otherwise(0).alias("draw"),
    col("away_score").alias("goal_scored"),
    col("home_score").alias("goals_conceded"),
    when(col("home_score") == 0, 1).otherwise(0).alias("clean_sheet"))
    
#marging both of them
unified_stat=home_stats_recent.union(away_stats_recent)

In [22]:
#preparing the recent master dataset to compare
from pyspark.sql.functions import lit
df_recent_master=unified_stat.groupBy("team").agg(
    lit(2026).alias("year"),
    spark_count("*").alias("played"),
    spark_sum("Win").alias("won"),
    spark_sum("loss").alias("loss"),
    spark_sum("draw").alias("drawn"),
    spark_sum("goal_scored").alias("goal_count"),
    spark_round(spark_avg(col("goal_scored")-col("goals_conceded")), 2).alias("goal_difference_avg"),
    spark_sum("clean_sheet").alias("clean_sheet"),
    spark_round((100 * spark_sum("clean_sheet") / spark_count("*")), 2).alias("clean_sheet_percentage"),
    spark_round(spark_avg("goals_conceded"), 2).alias("av_goal_consumed")
)
announced_48_teams = [
    "Australia", "Iran", "Iraq", "Japan", "Jordan", "Qatar", "Saudi Arabia", "South Korea", "Uzbekistan","Algeria", 
    "Cape Verde", "Ivory Coast", "DR Congo", "Egypt", "Ghana", "Morocco", "Senegal", "South Africa", "Tunisia",
    "Canada", "Curaçao", "Haiti", "Mexico", "Panama", "United States", "Argentina", "Brazil", "Colombia", 
    "Ecuador", "Paraguay", "Uruguay", "New Zealand","Austria", "Belgium", "Bosnia and Herzegovina", "Croatia", 
    "Czech Republic", "England", "France", "Germany", "Netherlands", "Norway", "Portugal", "Scotland", "Spain", "Sweden", 
    "Switzerland", "Turkey"
]
df_recent_master=df_recent_master.filter(col("team").isin(announced_48_teams))
df_recent_master.limit(2).toPandas()

,team,year,played,won,loss,drawn,goal_count,goal_difference_avg,clean_sheet,clean_sheet_percentage,av_goal_consumed
0,Paraguay,2026,32,11,12,9,29.0,-0.13,14,43.75,1.03
1,Senegal,2026,43,31,4,8,88.0,1.53,26,60.47,0.51


In [23]:
df_recent_master.count()

48

In [24]:
# Convert your list to a DataFrame to perform the join
from pyspark.sql.functions import col
announced_df = spark.createDataFrame([(t,) for t in announced_48_teams], ["announced_team"])

# Find the missing teams
missing_teams = announced_df.join(
    df_recent_master, 
    announced_df.announced_team == df_recent_master.team, 
    "left_anti"
)

missing_teams.show()

+--------------+
|announced_team|
+--------------+
+--------------+



In [25]:
#inserting the FIFA announce world cup official team members
#announced_players=spark.read.option("header",True).option("inferschema",True).csv(
    #"FIFA_project/official_squads_2026.csv")
#announced_players.limit(5).toPandas()

In [26]:
#creating new column for modified names
#from pyspark.sql.functions import concat_ws,element_at
#announced_players=announced_players.withColumn("Player",concat_ws(
    #" ",element_at(split(col("player_name")," "),-1),element_at(split(col("player_name")," "),1)))

In [27]:
#from pyspark.sql.functions import initcap
#announced_players=announced_players.withColumn("Player",initcap(col("Player")))

In [28]:
#announced_players.limit(2).toPandas()

##Decision making

In [29]:
master_historical_df.limit(1).toPandas()

,year,team,played,won,loss,drawn,goal_count,goal_difference_avg,clean_sheet,clean_sheet_percentage,av_goal_consumed,tournament_winner,highest_goal_scorer,highest_goal_count,total_team_goals,contribution
0,1990,Argentina,7,4,2,1,5,0.14,3,42.86,0.57,Germany,Claudio Caniggia,2,5,40.0


In [30]:
##extracting historical benchmarks
from pyspark.sql.functions import mean as spark_median
benchmarks=master_historical_df.select(
    spark_round(spark_median("goal_difference_avg"),4).alias("median_goal_diff_avg"),
    spark_round(spark_median("clean_sheet_percentage"),4).alias("median_clean_sheet_perc"),
    spark_round(spark_median("av_goal_consumed"),4).alias("median_goal_consumed_avg")
).collect()[0]
benchmarks

Row(median_goal_diff_avg=0.8731, median_clean_sheet_perc=44.0483, median_goal_consumed_avg=0.8214)

In [31]:
#saving values in variables
target_goal_diff=benchmarks["median_goal_diff_avg"]
target_clean_sheet_perc=benchmarks["median_clean_sheet_perc"]
target_goal_conceded=benchmarks["median_goal_consumed_avg"]
print(f"Historical Benchmarks -> GD Avg: {target_goal_diff}, Clean Sheet %: {target_clean_sheet_perc}, Goals Conceded Avg: {target_goal_conceded}")

Historical Benchmarks -> GD Avg: 0.8731, Clean Sheet %: 44.0483, Goals Conceded Avg: 0.8214


In [32]:
df_recent_master.limit(1).toPandas()

,team,year,played,won,loss,drawn,goal_count,goal_difference_avg,clean_sheet,clean_sheet_percentage,av_goal_consumed
0,Paraguay,2026,32,11,12,9,29.0,-0.13,14,43.75,1.03


In [33]:
#scoring the current season pre WC games
df_recent_master_scored=df_recent_master.withColumn("gd_score",when(col("goal_difference_avg")>=target_goal_diff,1
                                                        ).otherwise(0))\
                                            .withColumn("cs_score",when(col("clean_sheet_percentage")>=target_clean_sheet_perc,1
                                                        ).otherwise(0))\
                                            .withColumn("defence_score",when(col("av_goal_consumed")<=target_goal_conceded,1
                                                        ).otherwise(0))
#total score calculation
df_recent_master_scored = df_recent_master_scored.withColumn(
    "total_benchmark_score", 
    col("gd_score") + col("cs_score") + col("defence_score")
)

In [41]:
df_recent_master_scored.orderBy(desc("total_benchmark_score")).limit(10).toPandas()

,team,year,played,won,loss,drawn,goal_count,goal_difference_avg,clean_sheet,clean_sheet_percentage,av_goal_consumed,gd_score,cs_score,defence_score,total_benchmark_score
0,Algeria,2026,47,32,5,10,104.0,1.49,22,46.81,0.72,1,1,1,3
1,Senegal,2026,43,31,4,8,88.0,1.53,26,60.47,0.51,1,1,1,3
2,Ivory Coast,2026,43,28,8,7,79.0,1.16,23,53.49,0.67,1,1,1,3
3,Argentina,2026,37,31,4,2,80.0,1.78,26,70.27,0.38,1,1,1,3
4,Morocco,2026,48,38,2,8,100.0,1.71,33,68.75,0.38,1,1,1,3
5,Spain,2026,39,30,3,6,104.0,1.85,19,48.72,0.82,1,1,1,3
6,Iran,2026,37,27,5,5,83.0,1.49,18,48.65,0.76,1,1,1,3
7,Uzbekistan,2026,40,24,5,11,64.0,0.88,22,55.00,0.73,1,1,1,3
8,Japan,2026,37,27,5,5,110.0,2.27,19,51.35,0.70,1,1,1,3
9,England,2026,39,27,6,6,82.0,1.51,21,53.85,0.59,1,1,1,3


In [37]:
#displaying top predictions for semi-finalists
from pyspark.sql.functions import asc
df_recent_master_scored.orderBy(
    
    desc("total_benchmark_score"),
    desc("goal_difference_avg"),
    asc("av_goal_consumed"),
    desc("clean_sheet_percentage")
).limit(48).toPandas()

,team,year,played,won,loss,drawn,goal_count,goal_difference_avg,clean_sheet,clean_sheet_percentage,av_goal_consumed,gd_score,cs_score,defence_score,total_benchmark_score
0,Japan,2026,37,27,5,5,110.0,2.27,19,51.35,0.70,1,1,1,3
1,Spain,2026,39,30,3,6,104.0,1.85,19,48.72,0.82,1,1,1,3
2,Argentina,2026,37,31,4,2,80.0,1.78,26,70.27,0.38,1,1,1,3
3,Portugal,2026,38,28,6,4,98.0,1.76,18,47.37,0.82,1,1,1,3
4,Morocco,2026,48,38,2,8,100.0,1.71,33,68.75,0.38,1,1,1,3
5,Senegal,2026,43,31,4,8,88.0,1.53,26,60.47,0.51,1,1,1,3
6,England,2026,39,27,6,6,82.0,1.51,21,53.85,0.59,1,1,1,3
7,Algeria,2026,47,32,5,10,104.0,1.49,22,46.81,0.72,1,1,1,3
8,Iran,2026,37,27,5,5,83.0,1.49,18,48.65,0.76,1,1,1,3
9,Australia,2026,36,22,8,6,69.0,1.19,18,50.00,0.72,1,1,1,3


In [35]:
# Converting to Pandas and download as a single CSV file
df_recent_master_scored.toPandas().to_csv("world_cup_predictions_ready.csv", index=False)